# 🎓 AI Code Mentor — Flask API Server

This notebook runs the Flask API backend for the AI Code Mentor project.

**Steps:**
1. Install dependencies
2. Authenticate with HuggingFace
3. Upload project files
4. Start Flask API + ngrok tunnel
5. Copy the public URL → paste into your frontend

In [1]:
# Step 1: Mount Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

%pip install -q -U bitsandbytes accelerate transformers pylint \
    langchain langchain-community langchain-core \
    langchain-text-splitters langchain-huggingface \
    faiss-cpu rank_bm25 beautifulsoup4 huggingface_hub \
    sentence-transformers flask flask-cors

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.4/276.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [2]:
# Step 2: Authenticate with HuggingFace
import os
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HuggingFace')
    print('[OK] Got token from Colab Secrets')
except Exception:
    hf_token = input('Enter your HuggingFace token: ')

if hf_token:
    login(token=hf_token)
    os.environ['HF_TOKEN'] = hf_token  # Fixed environment variable name
    print('[OK] Logged in to HuggingFace')
else:
    print('[WARNING] No token provided')

[OK] Got token from Colab Secrets
[OK] Logged in to HuggingFace


## Step 3: Upload Project Files

Upload these 2 files to `/content/`:
- `code_reviewer.py`
- `api.py`

You can drag-and-drop them into the file browser on the left,
or run the cell below to upload interactively.

In [3]:
%%writefile code_reviewer.py
"""
AI Code Reviewer - Core Module
Combines RAG (Knowledge Base) + Static Analysis (Pylint) + LLM (Qwen2.5-Coder)
"""

import torch
import json
import re
import os
import subprocess
import tempfile
import textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# LangChain Imports
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter


# ---------------------------------------------------------
# 1. KNOWLEDGE BASE (RAG Engine)
# ---------------------------------------------------------
class KnowledgeBase:
    """
    RAG-based knowledge retrieval from a local offline Knowledge Base.
    Covers PEP 8 style, OWASP Security, and SOLID Design Principles.
    Uses hybrid search: FAISS (semantic) + BM25 (keyword).
    """

    def __init__(self, filepath="knowledge_base.md"):
        if not os.path.exists(filepath):
            raise FileNotFoundError(
                f"❌ Knowledge base file '{filepath}' not found! "
                "Please upload it to the /content/ folder on Colab."
            )

        print(f"📚 Loading Offline Knowledge Base from: {filepath}")
        loader = TextLoader(filepath, encoding='utf-8')
        data = loader.load()

        # Split on markdown section headers to preserve semantic meaning
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=60,
            separators=["\n## ", "\n### ", "\n- ", "\n", " "]
        )
        splits = text_splitter.split_documents(data)

        # CPU Embedding (Saves GPU for the LLM)
        embedding_model = HuggingFaceEmbeddings(
            model_name="all-MiniLM-L6-v2",
            model_kwargs={'device': 'cpu'}
        )

        self.vectorstore = FAISS.from_documents(splits, embedding_model)
        self.bm25_retriever = BM25Retriever.from_documents(splits)
        self.bm25_retriever.k = 3

        print(f"✅ Knowledge Base Ready. ({len(splits)} rule chunks indexed)")

    def search(self, queries):
        """
        Hybrid search: combines semantic (FAISS) and keyword (BM25) results.
        """
        results = []
        for q in queries:
            docs_faiss = self.vectorstore.similarity_search(q, k=2)
            docs_bm25 = self.bm25_retriever.invoke(q)
            results.extend(docs_faiss + docs_bm25)

        # Deduplicate
        unique_docs = {doc.page_content: doc for doc in results}.values()
        return "\n\n".join([f"[Guide Excerpt]: {doc.page_content}" for doc in unique_docs])


# ---------------------------------------------------------
# 2. THE UNIFIED REVIEWER (RAG + Pylint + LLM)
# ---------------------------------------------------------
class UnifiedCodeReviewer:
    """
    Main code review engine combining:
    - Static Analysis (Pylint)
    - RAG-based style guide lookup
    - LLM-powered deep analysis
    """

    def __init__(self, model_id="Qwen/Qwen2.5-Coder-7B-Instruct", hf_token=None):
        # 1. Initialize RAG
        self.kb = KnowledgeBase()

        print(f"🤖 Loading Brain: {model_id}...")

        # 2. Handle authentication
        if hf_token:
            from huggingface_hub import login
            login(token=hf_token)
        else:
            # Try to get token from environment or Kaggle
            hf_token = os.getenv("HF_TOKEN")
            if hf_token:
                from huggingface_hub import login
                login(token=hf_token)
            else:
                try:
                    from kaggle_secrets import UserSecretsClient
                    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
                    from huggingface_hub import login
                    login(token=hf_token)
                except Exception:
                    print("⚠️ No HF_TOKEN found. Model may fail to load if not cached.")

        # 3. Initialize Model (GPU with 4-bit quantization)
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
            low_cpu_mem_usage=True,
            torch_dtype=torch.float16
        )
        print("✅ System Online.")

    def _ask_llm(self, messages, max_tokens=200):
        """Generate response from the LLM using ChatML messages."""
        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.2,
            repetition_penalty=1.2,
            top_p=0.9,
            do_sample=True
        )
        # Decode and extract only the assistant's response
        full_output = self.tokenizer.decode(outputs[0], skip_special_tokens=False)
        # Split on the last assistant marker to get only the new response
        if "<|im_start|>assistant" in full_output:
            response = full_output.split("<|im_start|>assistant")[-1]
            response = response.replace("<|im_end|>", "").strip()
        else:
            response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            response = response[len(prompt):].strip()
        return response

    def _run_pylint(self, code_snippet):
        """Run static analysis on the code using Pylint."""
        with tempfile.NamedTemporaryFile(mode='w+', suffix='.py', delete=False) as f:
            f.write(code_snippet)
            fname = f.name

        try:
            result = subprocess.run(
                ["pylint", fname, "--output-format=json"],
                capture_output=True, text=True
            )

            if not result.stdout.strip():
                return "No Syntax Errors Found."

            try:
                errors = json.loads(result.stdout)
                # Limit to top 15 errors to prevent context flooding
                if len(errors) > 15:
                    report = [f"Line {e['line']}: {e['message']} ({e['symbol']})" for e in errors[:15]]
                    report.append("... (and more)")
                else:
                    report = [f"Line {e['line']}: {e['message']} ({e['symbol']})" for e in errors]
                return "\n".join(report)
            except json.JSONDecodeError:
                return "Pylint ran but output was not parseable."
        except Exception as e:
            return f"Pylint Failed to Run: {e}"
        finally:
            if os.path.exists(fname):
                os.remove(fname)

    def _generate_search_plan(self, code_snippet):
        """Analyze code to decide what style topics to search for (returns JSON)."""
        messages = [
            {"role": "system", "content": "You are a Python Style Analyst. Identify 3 SPECIFIC Python style topics relevant to the code. You MUST output ONLY valid JSON. Nothing else. No explanation. Schema: {\"queries\": [\"string\", \"string\", \"string\"]} Example: {\"queries\": [\"function naming conventions\", \"import placement\", \"list comprehensions\"]}"},
            {"role": "user", "content": f"Code:\n{code_snippet}"}
        ]

        raw_response = self._ask_llm(messages, max_tokens=150)

        try:
            json_match = re.search(r'\{.*\}', raw_response, re.DOTALL)
            if json_match:
                data = json.loads(json_match.group(0))
                queries = data.get("queries", [])
                # Validate: ensure we got a list of strings
                if isinstance(queries, list) and len(queries) > 0:
                    return queries[:3]
            return ["Python naming conventions", "code structure", "error handling"]
        except (json.JSONDecodeError, AttributeError):
            return ["Python naming conventions", "code structure", "error handling"]

    def review(self, code_snippet):
        """
        Main entry point: performs comprehensive code review.
        Returns formatted feedback string.
        """
        print("1. 🧠 Analyzing Code Logic...")
        linter_report = self._run_pylint(code_snippet)

        print("2. 🔍 Retrieving Style Rules...")
        search_terms = self._generate_search_plan(code_snippet)
        rag_rules = self.kb.search(search_terms)

        print("3. 📝 Generating Final Report...")
        system_msg = f"""You are a Python code reviewer. Review the student's code using the reports below.

STATIC ANALYSIS REPORT:
{linter_report}

STYLE GUIDE:
{rag_rules}

KNOWN BUG PATTERNS — flag these if you see them:
- Using eval() is a CRITICAL security vulnerability.
- Removing items from a list while iterating over it is a CRITICAL bug. Fix with list comprehension.
- Using open() without "with" statement means the file may never be closed.
- Bare "except:" without specifying an exception type hides all errors.
- Nested loops for searching can be replaced with sets for better performance.
- Hardcoded API keys, passwords, tokens, or secrets (e.g. "sk_live_", "ghp_", "password=") in source code is a CRITICAL security vulnerability. Always use environment variables.

RULES:
- Only report issues that ACTUALLY exist in the code. Do not make up problems.
- If the code is already correct and clean, say "No issues found" and return it unchanged.
- The refactored code MUST fix all bugs you identified. Do NOT return buggy code.
- Wrap refactored code in ```python and ``` markers.
- FORMATTING: Never add leading spaces or indentation before your top-level helper functions or main function in the code block. The `def` keyword must touch the left edge completely.

Use this exact format:

**1. Critical Issues:**
List bugs and security issues. If none exist, write "No critical issues found."

**2. Style Analysis:**
List naming or formatting issues based on PEP 8. If code is clean, say so.

**3. Refactored Solution:**
```python
(corrected code here)
```"""

        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": f"Review this code:\n\n{code_snippet}"}
        ]

        raw = self._ask_llm(messages, max_tokens=1024)
        return self._clean_code_blocks(raw)

    def _clean_code_blocks(self, text):
        """Strip leading whitespace/indentation inside ```python code blocks."""
        def dedent_block(match):
            code = match.group(1)
            return "```python\n" + textwrap.dedent(code).lstrip("\n") + "```"
        return re.sub(r"```python\n(.*?)```", dedent_block, text, flags=re.DOTALL)

    @staticmethod
    def extract_severity(review_text):
        """
        Parse the AI's review output to count critical vs. style issues.
        Used by the API to auto-grade submissions.

        Returns: { "critical_count": int, "style_count": int }
        """
        critical_count = 0
        style_count = 0

        # Split into sections
        sections = re.split(r'\*\*\d+\.\s*', review_text)

        for section in sections:
            lower = section.lower()

            # Count critical issues (in the Critical Issues section)
            if 'critical' in lower or 'issue' in lower or 'bug' in lower:
                if 'no critical issues' in lower or 'no issues found' in lower:
                    continue
                # Count bullet points / numbered items as individual issues
                bullets = re.findall(r'(?:^|\n)\s*[-•*]\s+', section)
                numbered = re.findall(r'(?:^|\n)\s*\d+[\.\)]\s+', section)
                count = len(bullets) + len(numbered)
                # At minimum 1 if the section has content beyond the header
                content = re.sub(r'[^a-zA-Z]', '', section)
                if count == 0 and len(content) > 20:
                    count = 1
                critical_count += count

            # Count style issues (in the Style Analysis section)
            elif 'style' in lower or 'pep' in lower or 'naming' in lower:
                if 'no style' in lower or 'code is clean' in lower or 'follows pep' in lower:
                    continue
                bullets = re.findall(r'(?:^|\n)\s*[-•*]\s+', section)
                numbered = re.findall(r'(?:^|\n)\s*\d+[\.\)]\s+', section)
                count = len(bullets) + len(numbered)
                content = re.sub(r'[^a-zA-Z]', '', section)
                if count == 0 and len(content) > 20:
                    count = 1
                style_count += count

        return {
            "critical_count": critical_count,
            "style_count": style_count
        }


Writing code_reviewer.py


In [4]:
%%writefile api.py
"""
AI Code Mentor - Flask REST API
Wraps code_reviewer.py as a REST API for the custom frontend.
Includes endpoints for review history, TA annotations, and dashboard analytics.
"""

from flask import Flask, request, jsonify
from flask_cors import CORS
import gc
import torch
import time

# Import core engine and database
from code_reviewer import UnifiedCodeReviewer
from database import init_db, register_student, get_student, save_review, \
    get_reviews_by_student, get_all_reviews, get_review_by_id, \
    save_annotation, get_annotations_for_review, \
    get_class_analytics, get_students_summary, calculate_grade
import os

# ==========================================
# Colab Drive Caching Setup
# ==========================================
if os.path.exists("/content/drive/MyDrive"):
    os.environ['HF_HOME'] = '/content/drive/MyDrive/huggingface_cache'
    print(f"📁 Google Drive Cache Enabled: {os.environ['HF_HOME']}")
elif os.path.exists("/content"):
    print("⚠️ Google Drive not mounted. Model will download to temporary Colab storage.")
    print("   Tip: Mount your drive using the folder icon on the left before running this cell to cache the 4GB download!")

# ==========================================
# Flask App Setup
# ==========================================
app = Flask(__name__)
CORS(app)  # Allow frontend from any origin

import threading

# Global engine reference (loaded once)
engine = None
engine_lock = threading.Lock()

def get_engine():
    """Load the AI engine (singleton pattern)."""
    global engine
    with engine_lock:
        if engine is None:
            print("🤖 Initializing AI Code Review Engine...")
            engine = UnifiedCodeReviewer()
            print("✅ Engine ready!")
    return engine


# ==========================================
# Core API Endpoints
# ==========================================

@app.route("/api/health", methods=["GET"])
def health():
    """Health check — returns model status."""
    return jsonify({
        "status": "online" if engine else "model_not_loaded",
        "model": "Qwen2.5-Coder-7B-Instruct (4-bit)",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"
    })


@app.route("/api/review", methods=["POST"])
def review():
    """
    Main review endpoint.
    Expects JSON: { "code": "def foo():...", "student_id": "250001" }
    Returns JSON: { "review": "...", "time": 12.3, "review_id": 1, "grade": 85 }
    """
    data = request.get_json()

    if not data or "code" not in data:
        return jsonify({"error": "Missing 'code' field in request body"}), 400

    code = data["code"].strip()
    student_id = data.get("student_id", "anonymous")

    if not code:
        return jsonify({"error": "Code cannot be empty"}), 400

    # Limit input size to prevent context overflow
    lines = code.split("\n")
    if len(lines) > 150:
        return jsonify({
            "error": f"Code too long ({len(lines)} lines). Maximum is 150 lines to ensure quality output."
        }), 400

    try:
        reviewer = get_engine()
        start = time.time()
        result = reviewer.review(code)
        elapsed = round(time.time() - start, 2)

        # Extract severity for grading
        severity = UnifiedCodeReviewer.extract_severity(result)
        critical_count = severity["critical_count"]
        style_count = severity["style_count"]
        grade = calculate_grade(critical_count, style_count)

        # Save to database
        saved = save_review(
            student_id=student_id,
            code_snippet=code,
            review_result=result,
            review_time_sec=elapsed,
            line_count=len(lines),
            critical_count=critical_count,
            style_count=style_count
        )

        # Cleanup GPU memory after review
        gc.collect()
        torch.cuda.empty_cache()

        return jsonify({
            "review": result,
            "time": elapsed,
            "lines": len(lines),
            "review_id": saved["id"],
            "grade": saved["grade"],
            "critical_count": critical_count,
            "style_count": style_count
        })

    except torch.cuda.OutOfMemoryError:
        gc.collect()
        torch.cuda.empty_cache()
        return jsonify({
            "error": "GPU out of memory. Try shorter code or restart the runtime."
        }), 503

    except Exception as e:
        return jsonify({"error": f"Review failed: {str(e)}"}), 500


@app.route("/api/load", methods=["POST"])
def load_model():
    """Trigger AI model loading in a background thread."""
    def background_task():
        try:
            get_engine()
        except Exception as e:
            print(f"Background load error: {e}")

    thread = threading.Thread(target=background_task)
    thread.start()
    return jsonify({"status": "loading_started"})


# ==========================================
# Student Endpoints
# ==========================================

@app.route("/api/student/register", methods=["POST"])
def register():
    """Register a student with their ID and name."""
    data = request.get_json()
    if not data or "student_id" not in data or "name" not in data:
        return jsonify({"error": "Missing 'student_id' or 'name'"}), 400

    student_id = data["student_id"].strip()
    name = data["name"].strip()

    if not student_id or not name:
        return jsonify({"error": "Student ID and name cannot be empty"}), 400

    result = register_student(student_id, name)
    return jsonify(result)


@app.route("/api/reviews/<student_id>", methods=["GET"])
def student_reviews(student_id):
    """Get all reviews for a specific student."""
    reviews = get_reviews_by_student(student_id)
    return jsonify({"reviews": reviews, "count": len(reviews)})


# ==========================================
# Staff Dashboard Endpoints
# ==========================================

@app.route("/api/reviews", methods=["GET"])
def all_reviews():
    """Get all reviews (staff dashboard). Supports pagination."""
    limit = request.args.get("limit", 50, type=int)
    offset = request.args.get("offset", 0, type=int)
    reviews = get_all_reviews(limit, offset)
    return jsonify({"reviews": reviews, "count": len(reviews)})


@app.route("/api/review/<int:review_id>", methods=["GET"])
def single_review(review_id):
    """Get a single review by ID."""
    review = get_review_by_id(review_id)
    if not review:
        return jsonify({"error": "Review not found"}), 404

    # Include annotations
    annotations = get_annotations_for_review(review_id)
    review["annotations"] = annotations
    return jsonify(review)


@app.route("/api/review/<int:review_id>/annotate", methods=["POST"])
def annotate_review(review_id):
    """Add a TA/Doctor annotation to a review."""
    data = request.get_json()
    if not data or "comment" not in data:
        return jsonify({"error": "Missing 'comment' field"}), 400

    staff_id = data.get("staff_id", "staff")
    comment = data["comment"].strip()

    if not comment:
        return jsonify({"error": "Comment cannot be empty"}), 400

    # Verify review exists
    review = get_review_by_id(review_id)
    if not review:
        return jsonify({"error": "Review not found"}), 404

    result = save_annotation(review_id, staff_id, comment)
    return jsonify(result), 201


@app.route("/api/review/<int:review_id>/annotations", methods=["GET"])
def review_annotations(review_id):
    """Get all annotations for a review."""
    annotations = get_annotations_for_review(review_id)
    return jsonify({"annotations": annotations, "count": len(annotations)})


@app.route("/api/dashboard/analytics", methods=["GET"])
def dashboard_analytics():
    """Get class-wide analytics for the staff dashboard."""
    analytics = get_class_analytics()
    return jsonify(analytics)


@app.route("/api/dashboard/students", methods=["GET"])
def dashboard_students():
    """Get student summary list for the staff dashboard."""
    students = get_students_summary()
    return jsonify({"students": students, "count": len(students)})


# ==========================================
# Launch
# ==========================================
if __name__ == "__main__":
    # Initialize database
    init_db()
    print("⏳ Pre-loading AI Model into GPU (this takes 2-3 minutes)...", flush=True)
    get_engine()
    print("🚀 Model loaded! Starting Flask API...", flush=True)
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)


Writing api.py


In [ ]:
# Upload project files (run this cell and select files)
from google.colab import files
import os

# Check if files already exist
needed = ['code_reviewer.py', 'api.py', 'knowledge_base.md']
missing = [f for f in needed if not os.path.exists(f'/content/{f}')]

if missing:
    print(f'Please upload: {missing}')
    uploaded = files.upload()
    for fname, content in uploaded.items():
        with open(f'/content/{fname}', 'wb') as f:
            f.write(content)
        print(f'[OK] Saved {fname}')
else:
    print('[OK] All files already present')

# Verify
for f in needed:
    status = '✅' if os.path.exists(f'/content/{f}') else '❌'
    print(f'{status} {f}')

In [7]:
# Step 5: Start Server & Tunnel (Regex Fix)
import os
import time
import subprocess
import urllib.request
import re

# 1. Start Cloudflare Tunnel
if not os.path.exists('cloudflared'):
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared')
    os.system('chmod +x cloudflared')

os.system('pkill -f cloudflared')
os.system('pkill -f api.py')
time.sleep(2)

print("🚀 Starting Cloudflare Tunnel...")
os.system('nohup ./cloudflared tunnel --url http://localhost:5000 > tunnel.log 2>&1 &')

# 2. Start Flask
print('Starting Flask API server (writing logs to flask_api.log)...')
log_file = open('/content/flask_api.log', 'w')
process = subprocess.Popen(['python', '/content/api.py'], stdout=log_file, stderr=subprocess.STDOUT)

# 3. Wait for GPU Load
print('⏳ Loading AI model... (Since it is cached, this should be fast!)')
ready = False
for i in range(180):
    if process.poll() is not None: break
    try:
        urllib.request.urlopen('http://localhost:5000/api/health')
        ready = True
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(5)

log_file.close()

# 4. Display Results
if ready and process.poll() is None:
    public_url = "URL_NOT_FOUND"

    # Use Regex to flawlessly extract the URL from the logs
    try:
        with open('tunnel.log', 'r') as f:
            log_content = f.read()
            # This looks for exactly: https://[anything].trycloudflare.com
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_content)
            if match:
                public_url = match.group(0)
    except Exception as e:
        print(f"Failed to read logs: {e}")

    print('\n\n' + '=' * 60)
    print(f'✅ API is live at: {public_url}')
    print('=' * 60)
    print('👉 Copy this URL and paste it into your frontend!')
else:
    print('\n\n[❌ ERROR] Server crashed! Here is the log:')
    print(os.popen('tail -n 30 /content/flask_api.log').read())

🚀 Starting Cloudflare Tunnel...
Starting Flask API server (writing logs to flask_api.log)...
⏳ Loading AI model... (Since it is cached, this should be fast!)
..................

✅ API is live at: https://sizes-congressional-medal-parker.trycloudflare.com
👉 Copy this URL and paste it into your frontend!


In [ ]:
!python evaluate_model.py


🔍 Reading test_codes.py...
✅ Successfully found 11 tests.
🚀 Found 11 test cases. Starting automated evaluation...

[1/11] Running TEST 1: Logic Bug — List mutation during iteration...
    Expected: 🔴 Critical issue detected
    ✅ Completed in 42.5s

[2/11] Running TEST 2: Security Issue — eval() and unclosed file...
    Expected: 🔴 Critical (eval is dangerous, file not closed)
    ✅ Completed in 53.3s

[3/11] Running TEST 3: Inefficient Algorithm — O(n²) nested loops...
    Expected: 🟡 Warning (use sets for O(n) lookup)
    ✅ Completed in 38.1s

[4/11] Running TEST 4: Bad Error Handling — bare except, unclosed file...
    Expected: 🔴 Critical (bare except), 🟡 Style (naming)
    ✅ Completed in 45.2s

[5/11] Running TEST 5: Clean Code — should get NO critical issues...
    Expected: 🟢 Good code, minimal or no issues
    ✅ Completed in 17.2s

[6/11] Running TEST 6: PEP-8 Style Violations...
    Expected: 🟡 Style (flag CamelCase, bad spacing)
    ✅ Completed in 44.5s

[7/11] Running TEST 7

In [ ]:
!python generate_report_metrics.py


 📊 ACADEMIC METRICS EVALUATION
✅ Code Resolution Rate (AST Validity): 100.0% (11/11)
🎯 Functional Error Detection Rate (Recall): 83.3% (5/6)
📜 Specification Compliance Rate: 100% (Detected PEP-8 naming correctly? True)

📈 Generating graphs for report...
✅ Saved 'report_metrics.png' successfully!

👉 Upload 'evaluate_model.py' to Colab, run it, then run this script to generate your report data!


In [ ]:
# (Optional) View server logs
# Run this cell to see recent API activity
import subprocess
result = subprocess.run(['tail', '-20', '/proc/' + str(process.pid) + '/fd/1'],
                       capture_output=True, text=True)
if result.stdout:
    print(result.stdout)
else:
    print('Server is running. Check the output of the previous cell for logs.')

In [ ]:
# Run this cell ONLY when you want to stop the server
import subprocess
import gc
import torch

subprocess.run(['pkill', '-f', 'api.py'], capture_output=True)
ngrok.kill()

gc.collect()
torch.cuda.empty_cache()
print('[OK] Server stopped and GPU memory cleared')